In [18]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [19]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-03-27 05:59:38--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 104.26.2.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.2’

book-crossings.zip. 100%[===================>]  24.88M   163MB/s    in 0.2s    

2026-03-27 05:59:38 (163 MB/s) - ‘book-crossings.zip.2’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [20]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [21]:
# Merge df_books and df_ratings
df = df_ratings.merge(df_books, on="isbn", how="left")

In [22]:
df.head(3)

,user,isbn,rating,title,author
0,276725,034545104X,0.0,Flesh Tones: A Novel,M. J. Rose
1,276726,0155061224,5.0,Rites of Passage,Judith Rae
2,276727,0446520802,0.0,The Notebook,Nicholas Sparks


In [23]:
# Display the number of rows and columns in the original DataFrame
df.shape

(1149780, 5)

In [24]:
# Count how many reviews each user has left
users_counts = df["user"].value_counts()

# Count how many reviews each book (ISBN) has received
isbn_counts = df["isbn"].value_counts()

# Keep only users with 200 or more reviews
users = users_counts[users_counts >= 200].index

# Keep only books with 100 or more reviews
isbn = isbn_counts[isbn_counts >= 100].index

In [25]:
# Create a new DataFrame containing only rows where user and ISBN are in the filtered lists
df = df.loc[(df["user"].isin(users.values)) & (df["isbn"].isin(isbn.values))]

In [26]:
# Drop duplicate reviews
df = df.drop_duplicates(['title', 'user'])

In [27]:
df.isna().sum()

,0
user,0
isbn,0
rating,0
title,238
author,238


In [28]:
# Dropping isbns without title and author
df = df.dropna(how='any')

In [29]:
df.shape

(49136, 5)

In [30]:
# Create a pivot table
df_pivot = df.pivot(index = 'title', columns = 'user', values = 'rating').fillna(0)
df_pivot.head(3)

user,254,2276,2766,2977,3363,4017,4385,6242,6251,6323,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [31]:
# Create a matrix
df_matrix = csr_matrix(df_pivot.values)

In [32]:
# Creating and training the model
nn = NearestNeighbors(metric='cosine')
nn.fit(df_matrix)

NearestNeighbors(metric='cosine')

In [34]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):
  # Create a list to put our model outputs into
  recommended_books = [book, []]

  # Pass through our book
  distance, book_info = nn.kneighbors([df_pivot.loc[book]], 6, return_distance=True)

  # Gather the text and distance & reverse the order
  recom_book_info = df_pivot.iloc[np.flip(book_info[0])[:-1]].index.to_list()
  recom_distance = list(np.flip(distance[0])[:-1])

  # for each value in our two variables append to our empty book list above
  for r in zip(recom_book_info, recom_distance):
    recommended_books[1].append(list(r))

  return recommended_books

In [35]:
# Test function
get_recommends('The Catcher in the Rye')

['The Catcher in the Rye',
 [['Tis: A Memoir', np.float32(0.78771865)],
  ["ANGELA'S ASHES", np.float32(0.77560645)],
  ['Their Eyes Were Watching God: A Novel', np.float32(0.7733084)],
  ['1984', np.float32(0.76593226)],
  ['To Kill a Mockingbird', np.float32(0.7657838)]]]

In [36]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", np.float32(0.8016211)], ['The Weight of Water', np.float32(0.77085835)], ['The Surgeon', np.float32(0.7699411)], ['I Know This Much Is True', np.float32(0.7677075)], ['The Lovely Bones: A Novel', np.float32(0.7234864)]]]
You passed the challenge! 🎉🎉🎉🎉
